In [1]:
from __future__ import annotations

import re
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
import torch

from transformers import AutoModel, AutoTokenizer

pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


In [2]:
PIPELINE_ROOT = Path("/mnt/primary/Finnhub Pipeline")
ANSWER_ROOT = PIPELINE_ROOT / "finnhub_answers"

START_DATE = pd.Timestamp("2026-07-15")
END_DATE = pd.Timestamp("2026-08-28")

MODEL_NAME = "bert-base-uncased"
MAX_TOKENS_PER_CHUNK = 510
CHUNK_BATCH_SIZE = 16

OUTPUT_ROOT = PIPELINE_ROOT / "answer_change_results" / "bert"
GRAPH_DIR = OUTPUT_ROOT / "daily_graphs"
EMBEDDING_CACHE = OUTPUT_ROOT / "bert_answer_embeddings.npz"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

print("Answer folder:", ANSWER_ROOT)
print("Period:", START_DATE.date(), "to", END_DATE.date())
print("Model:", MODEL_NAME)


Answer folder: /mnt/primary/Finnhub Pipeline/finnhub_answers
Period: 2026-07-15 to 2026-08-28
Model: bert-base-uncased


In [3]:
def extract_date(path: Path):
    for part in reversed(path.parts):
        try:
            date = pd.Timestamp(part).normalize()
            if START_DATE <= date <= END_DATE:
                return date
        except Exception:
            pass

    match = re.search(r"(\d{4}-\d{2}-\d{2})", path.stem)
    if match:
        date = pd.Timestamp(match.group(1)).normalize()
        if START_DATE <= date <= END_DATE:
            return date

    return None


def extract_company(path: Path) -> str:
    name = re.sub(r"_answer_\d{4}-\d{2}-\d{2}$", "", path.stem, flags=re.IGNORECASE)
    name = re.sub(r"_answer$", "", name, flags=re.IGNORECASE)
    return name.strip()


def read_answer(path: Path) -> str:
    text = path.read_text(encoding="utf-8", errors="replace")
    return re.sub(r"\s+", " ", text).strip()


if not ANSWER_ROOT.exists():
    raise FileNotFoundError(f"Answer folder not found: {ANSWER_ROOT}")

records = []

for path in ANSWER_ROOT.rglob("*.txt"):
    date = extract_date(path)

    if date is None:
        continue

    text = read_answer(path)

    if not text:
        continue

    records.append(
        {
            "Date": date,
            "Company": extract_company(path),
            "AnswerText": text,
            "AnswerPath": str(path),
        }
    )

answers = (
    pd.DataFrame(records)
    .drop_duplicates(subset=["Date", "Company"], keep="last")
    .sort_values(["Company", "Date"])
    .reset_index(drop=True)
)

if answers.empty:
    raise ValueError("No answer files were found in the selected period.")

print("Answers loaded:", len(answers))
print("Companies:", answers["Company"].nunique())
print("Date range:", answers["Date"].min().date(), "to", answers["Date"].max().date())


Answers loaded: 4400
Companies: 100
Date range: 2026-07-15 to 2026-08-28


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

print("Device:", device)


Device: cuda


In [5]:
def split_into_chunks(text: str) -> list[list[int]]:
    token_ids = tokenizer.encode(text, add_special_tokens=False, truncation=False)
    return [
        token_ids[start:start + MAX_TOKENS_PER_CHUNK]
        for start in range(0, len(token_ids), MAX_TOKENS_PER_CHUNK)
    ]


def mean_pool(hidden: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).float()
    return (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)


@torch.no_grad()
def embed_answer(text: str) -> np.ndarray:
    chunks = split_into_chunks(text)

    if not chunks:
        return np.zeros(model.config.hidden_size, dtype=np.float32)

    chunk_vectors = []

    for start in range(0, len(chunks), CHUNK_BATCH_SIZE):
        batch_chunks = chunks[start:start + CHUNK_BATCH_SIZE]
        prepared_ids = [
            tokenizer.prepare_for_model(chunk, add_special_tokens=True, truncation=False)["input_ids"]
            for chunk in batch_chunks
        ]

        batch = tokenizer.pad({"input_ids": prepared_ids}, padding=True, return_tensors="pt")
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        chunk_vectors.append(mean_pool(outputs.last_hidden_state, attention_mask).cpu().numpy())

    vector = np.vstack(chunk_vectors).mean(axis=0)
    norm = np.linalg.norm(vector)

    if norm > 0:
        vector = vector / norm

    return vector.astype(np.float32)


cache_loaded = False

if EMBEDDING_CACHE.exists():
    try:
        cached = np.load(EMBEDDING_CACHE, allow_pickle=True)
        cached_paths = cached["paths"].astype(str).tolist()
        current_paths = answers["AnswerPath"].astype(str).tolist()

        if cached_paths == current_paths:
            embedding_matrix = cached["embeddings"]
            cache_loaded = True
            print("Loaded cached embeddings:", embedding_matrix.shape)
    except Exception as exc:
        print("Could not use embedding cache:", exc)

if not cache_loaded:
    embeddings = []

    for index, row in answers.iterrows():
        embeddings.append(embed_answer(row["AnswerText"]))

        if (index + 1) % 25 == 0 or (index + 1) == len(answers):
            print(f"Embedded {index + 1:,}/{len(answers):,} answers")

    embedding_matrix = np.vstack(embeddings)
    np.savez_compressed(
        EMBEDDING_CACHE,
        paths=answers["AnswerPath"].astype(str).to_numpy(),
        embeddings=embedding_matrix,
    )

answers["EmbeddingIndex"] = np.arange(len(answers))


Token indices sequence length is longer than the specified maximum sequence length for this model (7954 > 512). Running this sequence through the model will result in indexing errors
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Embedded 25/4,400 answers
Embedded 50/4,400 answers
Embedded 75/4,400 answers
Embedded 100/4,400 answers
Embedded 125/4,400 answers
Embedded 150/4,400 answers
Embedded 175/4,400 answers
Embedded 200/4,400 answers
Embedded 225/4,400 answers
Embedded 250/4,400 answers
Embedded 275/4,400 answers
Embedded 300/4,400 answers
Embedded 325/4,400 answers
Embedded 350/4,400 answers
Embedded 375/4,400 answers
Embedded 400/4,400 answers
Embedded 425/4,400 answers
Embedded 450/4,400 answers
Embedded 475/4,400 answers
Embedded 500/4,400 answers
Embedded 525/4,400 answers
Embedded 550/4,400 answers
Embedded 575/4,400 answers
Embedded 600/4,400 answers
Embedded 625/4,400 answers
Embedded 650/4,400 answers
Embedded 675/4,400 answers
Embedded 700/4,400 answers
Embedded 725/4,400 answers
Embedded 750/4,400 answers
Embedded 775/4,400 answers
Embedded 800/4,400 answers
Embedded 825/4,400 answers
Embedded 850/4,400 answers
Embedded 875/4,400 answers
Embedded 900/4,400 answers
Embedded 925/4,400 answers
Embe

In [6]:
change_rows = []

for company, group in answers.groupby("Company"):
    group = group.sort_values("Date").reset_index(drop=True)

    if len(group) < 2:
        continue

    for i in range(1, len(group)):
        current_vector = embedding_matrix[int(group.loc[i, "EmbeddingIndex"])]
        previous_vector = embedding_matrix[int(group.loc[i - 1, "EmbeddingIndex"])]
        similarity = float(np.clip(np.dot(current_vector, previous_vector), -1.0, 1.0))

        change_rows.append(
            {
                "Company": company,
                "Date": group.loc[i, "Date"],
                "PreviousDate": group.loc[i - 1, "Date"],
                "BERT_Change": 1.0 - similarity,
            }
        )

daily_change = pd.DataFrame(change_rows)

if daily_change.empty:
    raise ValueError("No consecutive answer comparisons could be calculated.")

daily_change = daily_change.sort_values(["Company", "Date"]).reset_index(drop=True)


In [7]:
for company, group in daily_change.groupby("Company"):
    group = group.sort_values("Date").copy()

    peak_row = group.loc[group["BERT_Change"].idxmax()]
    peak_date = pd.Timestamp(peak_row["Date"])
    peak_change = float(peak_row["BERT_Change"])

    figure, axis = plt.subplots(figsize=(11, 5))
    axis.plot(group["Date"], group["BERT_Change"], marker="o")

    axis.set_xlim(START_DATE, END_DATE)
    axis.set_ylim(0.0, 1.0)
    axis.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
    axis.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))

    axis.set_xlabel("Date")
    axis.set_ylabel("BERT Change")
    axis.set_title(f"{company} — Daily Answer Change (BERT)")
    axis.tick_params(axis="x", rotation=45)

    axis.annotate(
        f"Highest spike: {peak_date.strftime('%d %b %Y')}",
        xy=(peak_date, min(peak_change, 1.0)),
        xytext=(10, -25),
        textcoords="offset points",
        arrowprops={"arrowstyle": "->"},
    )

    plt.tight_layout()

    safe_name = re.sub(r"[^A-Za-z0-9._-]+", "_", company).strip("_")
    figure.savefig(GRAPH_DIR / f"{safe_name}_bert_daily_change.png", dpi=150, bbox_inches="tight")
    plt.close(figure)

print("Saved daily graphs:", daily_change["Company"].nunique())
print("Graph folder:", GRAPH_DIR)


Saved daily graphs: 100
Graph folder: /mnt/primary/Finnhub Pipeline/answer_change_results/bert/daily_graphs


In [8]:
mean_change = (
    daily_change.groupby("Company", as_index=False)["BERT_Change"]
    .mean()
    .rename(columns={"BERT_Change": "Mean_BERT_Change"})
)

peak_rows = (
    daily_change.loc[daily_change.groupby("Company")["BERT_Change"].idxmax(), ["Company", "Date"]]
    .rename(columns={"Date": "Highest_Spike_Date"})
)

final_table = (
    mean_change.merge(peak_rows, on="Company", how="left")
    .sort_values("Mean_BERT_Change", ascending=False)
    .reset_index(drop=True)
)

final_table["Highest_Spike_Date"] = pd.to_datetime(final_table["Highest_Spike_Date"]).dt.strftime("%Y-%m-%d")
final_table.index = final_table.index + 1
final_table.index.name = "Rank"

display(final_table.round(4))


,Company,Mean_BERT_Change,Highest_Spike_Date
Rank,,,
1,CVS Health,0.0071,2026-07-20
2,Martin Marietta Materials,0.0066,2026-07-22
3,Adobe Inc,0.0066,2026-07-22
4,Accenture,0.0061,2026-07-23
5,"Nike, Inc",0.0061,2026-07-20
6,Iron Mountain,0.0061,2026-07-22
7,JPMorgan Chase,0.0061,2026-07-23
8,Amazon,0.0061,2026-07-22
9,Williams Companies,0.0060,2026-07-22


In [9]:
daily_change.to_csv(OUTPUT_ROOT / "bert_daily_change.csv", index=False)
final_table.to_csv(OUTPUT_ROOT / "bert_company_ranking.csv", index=True)

print("Saved results to:", OUTPUT_ROOT)


Saved results to: /mnt/primary/Finnhub Pipeline/answer_change_results/bert
